In [0]:
%run "./BRONZE"


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import os
import pandas as pd 
import datetime as dt
from pyspark.sql import SparkSession

In [0]:
BASE_PATH = "abfss://data@rgemrhealthcare.dfs.core.windows.net/"

BRONZE_PATH = BASE_PATH + "bronze/"
SILVER_PATH = BASE_PATH + "silver/"

%md
## Clean Departments

In [0]:
silver_Departments=(
    departments
    .withColumn('DeptID',trim('DeptID'))
    .withColumn('Name',trim('Name'))
    .withColumn('Location',trim('Location'))
    .withColumn('Specialization',trim('Specialization'))
    .filter(col('DeptID').isNotNull())
    .dropDuplicates(['DeptID'])
)
silver_Departments.count()

%md
## Clean Providers

In [0]:
silver_Providers = (
    providers
    .withColumn("ProviderID", trim(col("ProviderID")))
    .withColumn("ProviderName", trim(col("ProviderName")))
    .withColumn("Specialization", trim(col("Specialization")))
    .withColumn("DeptID", trim(col("DeptID")))
    .withColumn("Phone", trim(col("Phone")))
    .withColumn("Email", trim(col("Email")))
    .withColumn("HireDate", trim(col("HireDate")))
    .filter(col("ProviderID").isNotNull())
    .dropDuplicates(["ProviderID"])
)
silver_Providers.count()

%md
## Clean Patients

In [0]:
silver_patients=(

    patients
    .withColumn(('PatientID'),trim(col('PatientID')))
    .withColumn(('PatientName'),trim(col('PatientName')))
    .withColumn(('Gender'),trim(col('Gender')))
    .withColumn(('DateOfBirth'),trim(col('DateOfBirth')))
    .withColumn(('Phone'),trim(col('Phone')))
    .withColumn(('Email'),trim(col('Email')))
    .withColumn(('Address'),trim(col('Address')))
    .withColumn(('BloodGroup'),trim(col('BloodGroup')))
    .withColumn(('InsuranceType'),trim(col('InsuranceType')))
    .withColumn(('RegistrationDate'),trim(col('RegistrationDate')))
    .filter(col('PatientID').isNotNull())
    .dropDuplicates(['PatientID'])
)
silver_patients.count()

%md
## Clean ENCOUNTER

In [0]:
 silver_encounters=(
     encounters
     .withColumn('EncounterID',trim(col('EncounterID')))
     .withColumn('PatientID',trim(col('PatientID')))
     .withColumn('ProviderID',trim(col('ProviderID')))
     .withColumn(('EncounterDate'),to_date((col('EncounterDate')),'MM/dd/yyyy'))
     .withColumn('FollowUpDate',to_date('FollowUpDate','MM/dd/yyyy'))
     .withColumn('Status',trim(col('Status')))
     .withColumn('Diagnosis',trim(col('Diagnosis')))
     .withColumn('TreatmentAdvice',trim('TreatmentAdvice'))
     .filter(col('EncounterID').isNotNull())
     .dropDuplicates(['EncounterID'])
 )
 silver_encounters.count()
 

%md
## Clean TRANSACTION

In [0]:
silver_transaction=(

    transactions
    .withColumn('TransactionID',trim(col('TransactionID')))
    .withColumn('EncounterID',trim(col('EncounterID')))
    .withColumn('TransactionDate',to_date(col('TransactionDate'),'MM/dd/yyyy'))
    .withColumn('TransactionType',trim(col('TransactionType')))
    .withColumn('Amount',col('Amount').cast('double'))
    .withColumn('PaymentMethod',trim(col('PaymentMethod')))
    .filter(col('TransactionID').isNotNull())
    .dropDuplicates(['TransactionID'])
)
silver_transaction.count()
display(silver_transaction.limit(10))
